In [0]:

spark.conf.set("fs.azure.account.auth.type.<storage-account>.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.<storage-account>.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.<storage-account>.dfs.core.windows.net", "<application-id>")
spark.conf.set("fs.azure.account.oauth2.client.secret.<storage-account>.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.<storage-account>.dfs.core.windows.net", "https://login.microsoftonline.com/<directory-id>/oauth2/token")

In [0]:
df_sales_16 = spark.read.format("csv")\
          .option("header", "true")\
          .option("inferSchema", "true")\
          .load("abfs://bronze@adventureworksdatalakeom.dfs.core.windows.net/AdventureWorks_Sales_2016")

In [0]:
df_sales_16.display()

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2016-01-01,2002-10-17,SO48797,385,14335,1,1,1
2016-01-01,2002-09-30,SO48802,383,24923,9,1,1
2016-01-01,2002-11-29,SO48801,326,15493,1,1,1
2016-01-01,2002-11-16,SO48799,352,26708,4,1,1
2016-01-01,2002-12-16,SO48798,369,23332,9,1,1
2016-01-01,2002-12-02,SO48800,342,15491,5,1,1
2016-01-01,2002-10-19,SO48795,375,16538,8,1,1
2016-01-01,2002-11-23,SO48796,375,15094,7,1,1
2016-01-02,2002-12-01,SO48804,356,12276,8,1,1
2016-01-02,2002-09-12,SO48814,360,13647,9,1,1


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_sales_16 = df_sales_16.withColumn(
    "OrderNumber",
    regexp_replace(col("OrderNumber"), "S", "T")
)

# Agg

In [0]:
df_sales_16 = df_sales_16.groupBy("ProductKey").agg(
    sum("OrderQuantity").alias("TotalSold")
)

In [0]:
df_sales_16.display()

ProductKey,TotalSold
471,66
540,389
580,86
588,48
481,268
472,71
322,15
362,301
375,147
593,17


In [0]:
df_sales_16.write.format("parquet")\
            .mode("append")\
            .option("path","abfs://silver@adventureworksdatalakeom.dfs.core.windows.net/AdventureWorks_Sales_2016")\
            .save()